# GraspMamba setup - Colab / Kaggle GPU

Enable GPU and Internet first. Upload/clone the repository and selected data. See `NOTEBOOK_SETUP.md`. CUDA compatibility must be checked in your actual runtime; this notebook has not been executed on hosted GPU here.


In [ ]:
from pathlib import Path
import os
import sys
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ.setdefault('MAX_JOBS', '2')
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/Grasp_Mamba_TS'), Path('/kaggle/working/Grasp_Mamba_TS')]
REPO = next((p for p in candidates if (p / 'train_network.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Upload/clone repo, then set REPO to its Linux folder and rerun this cell')
os.chdir(REPO)
# Let the shell script use this notebook kernel's Python environment.
os.environ['PYTHON_BIN'] = sys.executable
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache' if Path('/kaggle').exists() else '/content/hf-cache')
print('Repo:', REPO, '\nHF_HOME:', os.environ['HF_HOME'])


In [ ]:
import sys
import torch
import subprocess
print(sys.version)
print('torch', torch.__version__, 'CUDA wheel', torch.version.cuda, 'GPU enabled', torch.cuda.is_available())
subprocess.run(['nvidia-smi'], check=False)
subprocess.run('nvcc --version', shell=True, check=False)


## Install packages
Keep the matched runtime torch/torchvision initially. Restart the kernel/session after this cell, then rerun the first cell. The optional fixed torch/CUDA profile and rebuild steps are in `NOTEBOOK_SETUP.md`.


In [ ]:
%pip install -r requirements-notebook.txt


## Install CUDA extensions
Run after package installation/restart, with CUDA-enabled torch already present. A matching wheel may be used; otherwise nvcc/toolkit is required for compilation. Restart again if previously imported modules were replaced.


In [ ]:
%pip install --no-build-isolation --no-deps -r requirements-cuda.txt


In [ ]:
!python tools/check_notebook_environment.py


## Cache pretrained models
About 1.73 GB of safetensors plus small files. Uses HF_HOME; excludes redundant weight copies.


In [ ]:
!python tools/download_pretrained.py


In [ ]:
!python tools/check_notebook_environment.py --model-smoke


## Baseline with the original Grasp-Anything++ loader
Use grasp-anywhere with data/grasp-anything (images, scene descriptions) and data/grasp-anything-pp (positive labels, instructions, part masks). Uses original ++ train/seen.obj filtering, then 90/10 train/validation, RGB mean subtraction, rotation/zoom and MSE. The upstream loader returns object text from scene_description; reading detailed grasp_instructions is commented out upstream and remains unchanged here. Shared trainer fixes remain.


In [ ]:
# Use Hugging Face IDs below, or replace with local Kaggle snapshot directories.
VISION_MODEL = "nvidia/MambaVision-L-1K"
TEXT_MODEL = "google/siglip-base-patch16-224"

DATASET_ROOT = REPO / 'data/grasp-anything'
BATCH_SIZE = 8
EPOCHS = 50
BATCHES_PER_EPOCH = 1000
LEARNING_RATE = 0.0001

!bash run_training.sh \
    --vision-model "{VISION_MODEL}" \
    --text-model "{TEXT_MODEL}" \
    --dataset grasp-anywhere \
    --dataset-path "{DATASET_ROOT}" \
    --add-file-path "{REPO / 'data/grasp-anything-pp'}" \
    --grasp-loss mse \
    --normalization legacy \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --batches-per-epoch {BATCHES_PER_EPOCH} \
    --lr {LEARNING_RATE} \
    --description original_loader_baseline


## Optional: prepared Grasp-Anything++ dataset
Change MANIFEST for a mounted Kaggle input. Preserve sibling relative paths from the prepared dataset.


In [ ]:
MANIFEST = REPO / 'data/grasp-anything-pp-ready/manifest.jsonl'
assert MANIFEST.is_file(), 'Set MANIFEST to the uploaded/mounted prepared manifest'
!python tools/check_grasp_anything_pp.py --manifest "{MANIFEST}"


## Optional: segmentation through run_training.sh

This cell explicitly selects the prepared ++ manifest loader with ImageNet normalization and Smooth L1. Defaults here run 10 optimizer updates for a quick check. Omitting --segmentation gives a comparison on the same prepared data; it does not select the original loader baseline above.


In [ ]:
# Use Hugging Face IDs below, or replace with local Kaggle snapshot directories.
VISION_MODEL = "nvidia/MambaVision-L-1K"
TEXT_MODEL = "google/siglip-base-patch16-224"

BATCH_SIZE = 1
EPOCHS = 1
BATCHES_PER_EPOCH = 10
LEARNING_RATE = 0.001
LAMBDA_SEG = 0.5

!bash run_training.sh \
    --vision-model "{VISION_MODEL}" \
    --text-model "{TEXT_MODEL}" \
    --dataset grasp-anything-language \
    --dataset-path "{MANIFEST}" \
    --segmentation \
    --mask-target part \
    --grasp-loss smooth_l1 \
    --normalization imagenet \
    --batch-size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --batches-per-epoch {BATCHES_PER_EPOCH} \
    --lr {LEARNING_RATE} \
    --lambda-seg {LAMBDA_SEG} \
    --description notebook_smoke


In [ ]:
!python -m pip freeze > requirements-colab-validated.txt
